In [ ]:
# import libraries

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
# load MovieLens

DATA_DIR = "../data/ml-latest-small"

ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")

print("Ratings:", len(ratings))
print("Users:", ratings["userId"].nunique())
print("Movies rated:", ratings["movieId"].nunique())

display(ratings.head())
display(movies.head())


Ratings: 100836
Users: 610
Movies rated: 9724


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# basic data check

print(ratings["rating"].describe())

print("\nMissing values in ratings:")
print(ratings.isnull().sum())

print("\nMissing values in movies:")
print(movies.isnull().sum())


count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64

Missing values in ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Missing values in movies:
movieId    0
title      0
genres     0
dtype: int64


In [ ]:
# split into training and test set

train, test = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

# Copies avoid pandas chained-assignment warnings.
train = train.copy()
test = test.copy()

print("Training ratings:", len(train))
print("Test ratings:", len(test))


Training ratings: 80668
Test ratings: 20168


In [ ]:
# map MovieLen IDs to model indices

user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user_to_index = {user_id: i for i, user_id in enumerate(user_ids)}
movie_to_index = {movie_id: i for i, movie_id in enumerate(movie_ids)}

num_users = len(user_to_index)
num_movies = len(movie_to_index)

train["user_index"] = train["userId"].map(user_to_index)
train["movie_index"] = train["movieId"].map(movie_to_index)

test["user_index"] = test["userId"].map(user_to_index)
test["movie_index"] = test["movieId"].map(movie_to_index)

print("Users:", num_users)
print("Movies:", num_movies)

display(train.head())


Users: 610
Movies: 9724


,userId,movieId,rating,timestamp,user_index,movie_index
80568,509,7347,3.0,1435994597,508,4285
50582,326,71462,4.0,1322252335,325,5629
8344,57,2115,3.0,965798155,56,134
99603,610,1127,4.0,1479544102,609,66
71701,462,2409,2.0,1174438249,461,1172


In [ ]:
# Convert movie genres into numeric features

movies = movies.copy()
movies["genre_list"] = movies["genres"].str.split("|")

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies["genre_list"])

genre_df = pd.DataFrame(
    genre_matrix,
    columns=mlb.classes_,
    index=movies["movieId"]
)

print("Genres:")
print(mlb.classes_)
print("Genre matrix shape:", genre_matrix.shape)


Genres:
['(no genres listed)' 'Action' 'Adventure' 'Animation' 'Children' 'Comedy'
 'Crime' 'Documentary' 'Drama' 'Fantasy' 'Film-Noir' 'Horror' 'IMAX'
 'Musical' 'Mystery' 'Romance' 'Sci-Fi' 'Thriller' 'War' 'Western']
Genre matrix shape: (9742, 20)


In [ ]:
# Prepare model inputs

X_train = train[["user_index", "movie_index"]].values
y_train = train["rating"].values.astype("float32")

X_test = test[["user_index", "movie_index"]].values
y_test = test["rating"].values.astype("float32")

X_train_genres = genre_df.loc[
    train["movieId"]
].values.astype("float32")

X_test_genres = genre_df.loc[
    test["movieId"]
].values.astype("float32")

hybrid_X_train = {
    "user": X_train[:, 0],
    "movie": X_train[:, 1],
    "genres": X_train_genres
}

hybrid_X_test = {
    "user": X_test[:, 0],
    "movie": X_test[:, 1],
    "genres": X_test_genres
}

print("User/movie training shape:", X_train.shape)
print("Genre training shape:", X_train_genres.shape)


User/movie training shape: (80668, 2)
Genre training shape: (80668, 20)


In [ ]:
# Define model class

class HybridMovieRecommender(keras.Model):
    def __init__(self, num_users, num_movies, num_genres, embedding_size=50):
        super().__init__()

        self.user_embedding = layers.Embedding(
            input_dim=num_users,
            output_dim=embedding_size
        )

        self.movie_embedding = layers.Embedding(
            input_dim=num_movies,
            output_dim=embedding_size
        )

        self.user_bias = layers.Embedding(
            input_dim=num_users,
            output_dim=1
        )

        self.movie_bias = layers.Embedding(
            input_dim=num_movies,
            output_dim=1
        )

        self.genre_dense = layers.Dense(
            embedding_size,
            activation="relu"
        )

    def call(self, inputs):
        user_indices = inputs["user"]
        movie_indices = inputs["movie"]
        genres = inputs["genres"]

        user_vector = self.user_embedding(user_indices)
        movie_vector = self.movie_embedding(movie_indices)

        user_bias = self.user_bias(user_indices)
        movie_bias = self.movie_bias(movie_indices)

        genre_vector = self.genre_dense(genres)

        combined_movie = movie_vector + genre_vector

        interaction = tf.reduce_sum(
            user_vector * combined_movie,
            axis=1
        )

        return (
            interaction
            + tf.squeeze(user_bias, axis=1)
            + tf.squeeze(movie_bias, axis=1)
        )


In [ ]:
# Create and compile model

hybrid_model = HybridMovieRecommender(
    num_users=num_users,
    num_movies=num_movies,
    num_genres=X_train_genres.shape[1],
    embedding_size=50
)

hybrid_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mean_squared_error",
    metrics=["mean_absolute_error"]
)


In [ ]:
# train with early stopping

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = hybrid_model.fit(
    hybrid_X_train,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping]
)

print("Epochs trained:", len(history.history["loss"]))


Epoch 1/30
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 2.6934 - mean_absolute_error: 1.2109 - val_loss: 0.8662 - val_mean_absolute_error: 0.7242
Epoch 2/30
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.7499 - mean_absolute_error: 0.6691 - val_loss: 0.7758 - val_mean_absolute_error: 0.6851
Epoch 3/30
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6673 - mean_absolute_error: 0.6272 - val_loss: 0.7622 - val_mean_absolute_error: 0.6735
Epoch 4/30
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6187 - mean_absolute_error: 0.6020 - val_loss: 0.7503 - val_mean_absolute_error: 0.6667
Epoch 5/30
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.5729 - mean_absolute_error: 0.5778 - val_loss: 0.7560 - val_mean_absolute_error: 0.6706
Epoch 6/30
1135/1135 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.5310 - mean_absolute_error: 0.5537 - val_loss: 0.7513 - val_mean_absolute_error: 0.6690
Epochs trained: 6


In [ ]:
# evaluate test set

test_loss, test_mae = hybrid_model.evaluate(
    hybrid_X_test,
    y_test
)

print("Hybrid model test loss:", test_loss)
print("Hybrid model test MAE:", test_mae)


631/631 ━━━━━━━━━━━━━━━━━━━━ 0s 268us/step - loss: 0.7504 - mean_absolute_error: 0.6638
Hybrid model test loss: 0.7504091858863831
Hybrid model test MAE: 0.6637604832649231


In [ ]:
# generate recs for existing MovieLens user

def recommend_movies(user_id, model, ratings, movies, top_n=10):
    if user_id not in user_to_index:
        raise ValueError("This user ID is not present in the MovieLens training data.")

    user_ratings = ratings[ratings["userId"] == user_id]
    watched_movies = set(user_ratings["movieId"])

    candidates = movies[
        (~movies["movieId"].isin(watched_movies))
        & (movies["movieId"].isin(movie_to_index))
    ].copy()

    candidates["movie_index"] = candidates["movieId"].map(movie_to_index)

    candidate_genres = genre_df.loc[
        candidates["movieId"]
    ].values.astype("float32")

    user_index = user_to_index[user_id]

    prediction_input = {
        "user": np.full(len(candidates), user_index, dtype=np.int32),
        "movie": candidates["movie_index"].values.astype(np.int32),
        "genres": candidate_genres
    }

    predictions = model.predict(prediction_input, verbose=0)
    candidates["predicted_rating"] = predictions

    return (
        candidates
        .sort_values("predicted_rating", ascending=False)
        [["movieId", "title", "genres", "predicted_rating"]]
        .head(top_n)
    )


In [ ]:
# provide recommendations 

recommendations = recommend_movies(
    user_id=1,
    model=hybrid_model,
    ratings=ratings,
    movies=movies,
    top_n=10
)

display(recommendations)


,movieId,title,genres,predicted_rating
6345,49265,Shooting Dogs (a.k.a. Beyond the Gates) (2005),Documentary|Drama|War,5.563243
7249,74226,"Dream of Light (a.k.a. Quince Tree Sun, The) (...",Documentary|Drama,5.449158
5842,32456,"Pom Poko (a.k.a. Raccoon War, The) (Heisei tan...",Animation|Comedy|Drama|Fantasy,5.440179
924,1223,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi,5.403619
8839,132333,Seve (2014),Documentary|Drama,5.394272
3294,4454,More (1998),Animation|Drama|Sci-Fi|IMAX,5.379263
8301,106642,"Day of the Doctor, The (2013)",Adventure|Drama|Sci-Fi,5.344554
5621,27156,Neon Genesis Evangelion: The End of Evangelion...,Action|Animation|Drama|Fantasy|Sci-Fi,5.343199
8802,130482,Too Late for Tears (1949),Crime|Drama|Film-Noir|Mystery|Thriller,5.314899
2453,3266,Man Bites Dog (C'est arrivé près de chez vous)...,Comedy|Crime|Drama|Thriller,5.296793


In [ ]:
# compare to what user previously liked

user_id = 1

liked_movies = (
    ratings[ratings["userId"] == user_id]
    .merge(movies[["movieId", "title", "genres"]], on="movieId")
    .sort_values("rating", ascending=False)
)

display(
    liked_movies[
        ["title", "genres", "rating"]
    ].head(10)
)


,title,genres,rating
231,M*A*S*H (a.k.a. MASH) (1970),Comedy|Drama|War,5.0
185,Excalibur (1981),Adventure|Fantasy,5.0
89,Indiana Jones and the Last Crusade (1989),Action|Adventure,5.0
90,Pink Floyd: The Wall (1982),Drama|Musical,5.0
190,From Russia with Love (1963),Action|Adventure|Thriller,5.0
189,Goldfinger (1964),Action|Adventure|Thriller,5.0
188,"Dirty Dozen, The (1967)",Action|Drama|War,5.0
186,Gulliver's Travels (1939),Adventure|Animation|Children,5.0
184,American Beauty (1999),Drama|Romance,5.0
179,"South Park: Bigger, Longer and Uncut (1999)",Animation|Comedy|Musical,5.0
